# 02.03 — Indexer le dataset de vision (sans imitation)

Prépare un index **de captures synchronisées** sans inventer d’actions humaines ni de masques sémantiques. Le PPO en ligne n’a PAS besoin de ce fichier pour apprendre à conduire. La séparation est faite par session, pour éviter une fuite entre images voisines.

In [4]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github 
Code : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code 
Python : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [5]:
import json, numpy as np, pandas as pd
sessions=sorted((PROJECT/'datasets').glob('session_*/frames.jsonl'))
print('Sessions camera trouvees:',len(sessions))
records=[]
for path in sessions:
    with path.open(encoding='utf8') as f:
        for line in f:
            if not line.strip():continue
            item=json.loads(line)
            image_path=path.parent/item['image']
            if not image_path.is_file():continue
            if abs(float(item.get('camera_gap_s',999)))>.40:continue
            records.append({'session':path.parent.name,'image_path':str(image_path),
               'camera_time':item.get('camera_time'),'telemetry_time':item.get('telemetry_time'),
               'speed':item.get('speed'),'lap':item.get('lap'),'camera_gap_s':item.get('camera_gap_s')})
if not records:
    print('Aucun couple valide: lancer 04_Capturer_Images.ipynb avec la camera active.')
else:
    table=pd.DataFrame(records).sort_values(['session','camera_time']).reset_index(drop=True)
    n_sessions=table.session.nunique()
    if n_sessions>=2:
        unique_sessions=sorted(table.session.unique())
        ntr=max(1,int(len(unique_sessions)*.8))
        train_sessions=set(unique_sessions[:ntr]);val_sessions=set(unique_sessions[ntr:])
        table['split']=table.session.map(lambda s:'train' if s in train_sessions else 'validation')
    else:
        # Une seule session: garder explicitement la validation vide plutot que fuir les donnees.
        table['split']='train'
        print('ATTENTION: une seule session. Collecter une autre session pour une vraie validation.')
    target=PROJECT/'datasets'/'index_images_synchronisees.csv'
    table.to_csv(target,index=False)
    print('Index enregistre:',target,'lignes:',len(table))
    display(table.groupby(['session','split']).size().reset_index(name='images'))

Sessions camera trouvees: 1
ATTENTION: une seule session. Collecter une autre session pour une vraie validation.
Index enregistre: C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\datasets\index_images_synchronisees.csv lignes: 90


,session,split,images
0,session_20261116_201727,train,90
